<a href="https://colab.research.google.com/github/Lima-Developer/machine-learning-cheat-analysis/blob/master/Algoritmo_Gen%C3%A9tico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import itertools
import random
import copy


class GeneticAlgorithm:

    def __init__(self, num_locations=12):
        self.population_size = 50
        self.num_locations = num_locations
        self.distance_matrix = [[0 for _ in range(self.num_locations)]
                                for _ in range(self.num_locations)]
        self._fill_matrix_random()
        self.optimal_solution = None
        self.population = []

    def _fill_matrix_random(self):
        """Preenche a matriz de distâncias aleatória e simétrica"""
        for i in range(self.num_locations):
            for j in range(i + 1, self.num_locations):
                distance = random.randint(10, 100)
                self.distance_matrix[i][j] = distance
                self.distance_matrix[j][i] = distance

    def generate_initial_population(self):
        """Gera população inicial aleatória"""
        self.population = []
        vertices = list(range(1, self.num_locations))

        for _ in range(self.population_size):
            random.shuffle(vertices)
            individual = [0] + vertices
            self.population.append(individual)

    def fitness(self, individual):
        """Função fitness - quanto menor a distância, maior o fitness"""
        total_distance = self.calculate_sum(individual)
        return 1 / (1 + total_distance)

    def calculate_sum(self, caminho):
        """Calcula distância total do caminho"""
        total_distance = 0
        for i in range(len(caminho) - 1):
            total_distance += self.distance_matrix[caminho[i]][caminho[i + 1]]

        # Volta ao ponto de partida
        last_client = caminho[-1]
        total_distance += self.distance_matrix[last_client][0]

        return total_distance

    def selection_by_raffle(self, evaluations):
        """Seleção por roleta"""
        total_fitness = sum(f for _, _, f in evaluations)
        pick = random.uniform(0, total_fitness)

        current = 0
        for caminho, distancia, fitness in evaluations:
            current += fitness
            if current >= pick:
                return caminho

    def crossover(self, parent1, parent2, start=1, end=3):
        """Order Crossover (OX)"""
        p1_clients = parent1[1:]
        p2_clients = parent2[1:]
        size = len(p1_clients)

        child_clients = [None] * size
        child_clients[start:end + 1] = p1_clients[start:end + 1]

        parent2_elements = [
            gene for gene in p2_clients if gene not in child_clients
        ]

        idx = 0
        for i in range(size):
            if child_clients[i] is None:
                child_clients[i] = parent2_elements[idx]
                idx += 1

        return [0] + child_clients

    def mutation(self, individual, mutation_rate=0.0005):
        """Mutação por troca de posições - CORRIGIDA"""
        if random.random() < mutation_rate:
            # Gera dois índices distintos (excluindo o primeiro que é o depósito)
            idx1, idx2 = random.sample(range(1, len(individual)), 2)
            # CORREÇÃO: idx1 e idx2 já são inteiros, não precisamos indexá-los
            individual[idx1], individual[idx2] = individual[idx2], individual[
                idx1]
        return individual

    def evaluate_population(self):
        """Avalia toda a população"""
        evaluations = []
        for individual in self.population:
            distancia = self.calculate_sum(individual)
            fit = self.fitness(individual)
            evaluations.append((individual, distancia, fit))
        evaluations.sort(key=lambda x: x[2], reverse=True)
        return evaluations

    def calculate_optimal_solution(self):
        """Calcula solução ótima por força bruta"""
        vertices = list(range(1, self.num_locations))
        best_distance = float('inf')
        best_path = None

        for perm in itertools.permutations(vertices):
            caminho = [0] + list(perm)
            distancia = self.calculate_sum(caminho)
            if distancia < best_distance:
                best_distance = distancia
                best_path = caminho

        self.optimal_solution = (best_path, best_distance)
        return best_path, best_distance

    def run(self, generations=100, mutation_rate=0.01):
        """Executa o algoritmo genético com taxa de mutação de 1% (mutation_rate)"""
        self.generate_initial_population()

        best_solutions = []

        for generation in range(generations):
            evaluations = self.evaluate_population()
            best_solutions.append(evaluations[0])

            new_population = []

            # Elitismo
            elite_size = max(1, self.population_size // 10)
            new_population.extend(
                [ind for ind, _, _ in evaluations[:elite_size]])

            # Preenche o restante
            while len(new_population) < self.population_size:
                parent1 = self.selection_by_raffle(evaluations)
                parent2 = self.selection_by_raffle(evaluations)

                child = self.crossover(parent1, parent2)
                child = self.mutation(child, mutation_rate)

                new_population.append(child)

            self.population = new_population

            if generation % 20 == 0:
                best_dist = evaluations[0][1]
                print(f"Geração {generation}: Melhor distância = {best_dist}")

        final_evaluation = self.evaluate_population()
        return final_evaluation[0], best_solutions


# Exemplo de uso
if __name__ == "__main__":
    ga = GeneticAlgorithm(num_locations=12)

    optimal_path, optimal_dist = ga.calculate_optimal_solution()
    print(f"Solução ótima: {optimal_path}, Distância: {optimal_dist}")

    best_solution, history = ga.run(generations=100)
    print(f"Melhor solução do AG: {best_solution[0]}, Distância: {best_solution[1]}")
